# Term bank EDA (stage 1)

Sources: Darby's own glossary, GIST (`Jerry999/multilingual-terminology`, Arabic.csv) and the Microsoft glossary (`ymoslem/MS-Glossary-EN-AR`), at the revisions pinned in `data/manifest.json`. Nothing here changes the raw files.

In [1]:
import json, re, sys
from collections import Counter
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from data.pipeline import read_source, verify_raw, REPORTS
from data.text import (arabic_share, is_latin_only, keeps_acronym_only, split_packed, acronym_kinds, is_acronym_term,
                       normalise_en, en_key, DIACRITIC, TATWEEL, PRESENTATION_FORM, EASTERN_DIGIT, HTML_TAG, EXTRA_WS)
from server.terms import Term, TermFinder
verify_raw()
REPORTS.mkdir(parents=True, exist_ok=True)
pd.set_option('display.width', 160); pd.set_option('display.max_colwidth', 50)
S = {}  # summary numbers for the report

## 1. Load and align columns

Every source becomes `source, row_id, en, ar`. Our glossary uses `term_en`/`term_msa`; GIST `English`/`Arabic`; MS `text_en`/`text_ar` (the card's other column names do not exist in the file).

In [2]:
g = read_source('gist'); m = read_source('ms'); o1 = read_source('darby_glossary'); o2 = read_source('darby_computing')
frames = [
    pd.DataFrame({'source': 'darby', 'row_id': o1.index.astype(str), 'en': o1.term_en, 'ar': o1.term_msa}),
    pd.DataFrame({'source': 'darby', 'row_id': ('c' + o2.index.astype(str)), 'en': o2.term_en, 'ar': o2.term_msa}),
    pd.DataFrame({'source': 'gist', 'row_id': g['Index'].astype(str), 'en': g.English, 'ar': g.Arabic}),
    pd.DataFrame({'source': 'ms', 'row_id': m.index.astype(str), 'en': m.text_en, 'ar': m.text_ar}),
]
df = pd.concat(frames, ignore_index=True).fillna('')
counts = df.groupby('source').size().rename('rows')
S['rows'] = counts.to_dict(); S['rows_total'] = int(len(df))
counts

source
darby       40
gist      4844
ms       22061
Name: rows, dtype: int64

## 2. Nulls and empty strings

In [3]:
empty = df.assign(en_empty=df.en.str.strip() == '', ar_empty=df.ar.str.strip() == '').groupby('source')[['en_empty', 'ar_empty']].sum()
S['empty'] = {k: {c: int(v) for c, v in r.items()} for k, r in empty.iterrows()}
empty

,en_empty,ar_empty
source,,
darby,0,0
gist,0,0
ms,0,0


## 3. Duplicates

Exact, case-insensitive, and after the matching key (`en_key`: lower case, singular, hyphens as spaces, acronyms keep case). Within each source and across sources.

In [4]:
df['en_norm'] = df.en.map(normalise_en)
df['key'] = df.en.map(en_key)
dup = {}
for src, d in df.groupby('source'):
    dup[src] = {
        'exact_pairs': int(d.duplicated(['en', 'ar']).sum()),
        'exact_en': int(d.en.duplicated().sum()),
        'case_insensitive_en': int(d.en.str.lower().duplicated().sum()),
        'same_key': int(d.key.duplicated().sum()),
    }
S['duplicates_within'] = dup
keys_by_src = {s: set(d.key) for s, d in df.groupby('source')}
S['overlap'] = {
    'darby_gist': len(keys_by_src['darby'] & keys_by_src['gist']),
    'darby_ms': len(keys_by_src['darby'] & keys_by_src['ms']),
    'gist_ms': len(keys_by_src['gist'] & keys_by_src['ms']),
    'all_three': len(keys_by_src['darby'] & keys_by_src['gist'] & keys_by_src['ms']),
}
S['unique_keys_all'] = int(df.key.nunique())
display(pd.DataFrame(dup).T); S['overlap'], S['unique_keys_all']

,exact_pairs,exact_en,case_insensitive_en,same_key
darby,0,0,0,0
gist,0,0,0,16
ms,1082,1964,2422,2642


({'darby_gist': 12, 'darby_ms': 27, 'gist_ms': 203, 'all_three': 7}, 24052)

In [5]:
# Keys that differ only by case are kept apart on purpose (IT vs it). How often does that happen?
lower_groups = df.groupby(df.key.str.lower()).key.nunique()
case_split = lower_groups[lower_groups > 1]
S['case_split_keys'] = int(len(case_split))
df[df.key.str.lower().isin(case_split.index[:12])][['source', 'en', 'ar']].sort_values('en').head(24)

,source,en,ar
71,gist,ARENA,ARENA
23322,ms,Arena,ساحة
6889,ms,AutoCorrect,تصحيح تلقائي
16522,ms,AutoFill,تعبئة تلقائية
23190,ms,AutoSave,حفظ تلقائي
21078,ms,BACK button,زر BACK
95,gist,BIO,BIO
21785,ms,Back button,"الزر ""رجوع"""
169,gist,CLIP,CLIP
6127,ms,EVERYONE,الجميع


## 4. Untranslated rows

Arabic empty, identical to the English, Latin script only, or only the English acronym with no Arabic word.

In [6]:
df['ar_same_as_en'] = df.ar.str.strip().str.lower() == df.en.str.strip().str.lower()
df['ar_latin_only'] = df.ar.map(is_latin_only)
df['ar_acronym_only'] = df.ar.map(keeps_acronym_only)
df['ar_mixed_script'] = df.ar.map(lambda s: 0 < arabic_share(s) < 1)
untr = df.groupby('source')[['ar_same_as_en', 'ar_latin_only', 'ar_acronym_only', 'ar_mixed_script']].sum()
S['untranslated'] = {k: {c: int(v) for c, v in r.items()} for k, r in untr.iterrows()}
display(untr)
df[df.ar_latin_only | df.ar_mixed_script][['source', 'en', 'ar']].head(25)

,ar_same_as_en,ar_latin_only,ar_acronym_only,ar_mixed_script
source,,,,
darby,0,0,0,0
gist,60,60,37,83
ms,16,45,8,2636


,source,en,ar
64,gist,A * algorithm,خوارزمية A*
65,gist,A/B test,اختبار A/B
66,gist,A2C,A2C
67,gist,AI,AI
69,gist,API,API
70,gist,AQA,AQA
71,gist,ARENA,ARENA
81,gist,AmbigQA,AmbigQA
91,gist,BART,BART
93,gist,BERTs,BERTs


## 5. Acronyms and short forms in English

In [7]:
df['acr_kinds'] = df.en.map(acronym_kinds)
df['is_acronym'] = df.en.map(is_acronym_term)
kind_counts = Counter(k for ks in df.acr_kinds for k in ks)
S['acronym_kinds'] = dict(kind_counts)
S['acronym_terms'] = df.groupby('source').is_acronym.sum().astype(int).to_dict()
print(kind_counts)
top = df[df.is_acronym].assign(n=1).groupby('en').agg(n=('n', 'sum'), sources=('source', lambda s: ','.join(sorted(set(s)))), ar=('ar', 'first'))
top = top.sort_values(['n', 'en'], ascending=[False, True]).head(50)
S['acronym_top50'] = top.reset_index()[['en', 'sources', 'ar']].to_dict('records')
top

Counter({'all_caps': 1397, 'digit_mix': 171, 'slash': 8, 'dots': 7, 'acronym_plural': 6})


,n,sources,ar
en,,,
IM,4,ms,رسالة فورية
AI,3,"darby,gist,ms",الذكاء الاصطناعي
A/B test,2,"gist,ms",اختبار A/B
API,2,"darby,gist",واجهة برمجة التطبيقات
CDA,2,"gist,ms",CDA
DSS,2,ms,DDS
MFA,2,ms,مصادقة متعددة العوامل (MFA)
RSVP,2,ms,طلب الاستجابة
TV,2,ms,التلفزيون


In [8]:
# Acronyms whose Arabic keeps the acronym only, or is Latin only: the core of the stage 3 work queue.
acr = df[df.is_acronym]
S['acronyms_without_arabic_meaning'] = int((acr.ar_latin_only | acr.ar_acronym_only).sum())
S['acronyms_with_mixed_arabic'] = int(acr.ar_mixed_script.sum())
acr[['source', 'en', 'ar']].sample(min(20, len(acr)), random_state=0)

,source,en,ar
10729,ms,OLEDB,OLE DB
95,gist,BIO,BIO
17782,ms,CC,شرح مكتوب لأحداث العرض
23137,ms,PPM,إدارة قوائم المشاريع (PPM)
6342,ms,FAQ,الأسئلة المتداولة
22976,ms,RMA,تفويض إرجاع البضائع (RMA)
21154,ms,ZAR,راند جنوب أفريقي
21152,ms,TRY,ليرة تركية
25868,ms,ACE,تنفيذ تعليمات برمجية تحكّمية
7912,ms,SCL,مستوى الثقة في البريد العشوائي


## 6. Script analysis of the Arabic values

In [9]:
script = pd.DataFrame({
    'source': df.source,
    'diacritics': df.ar.str.contains(DIACRITIC),
    'tatweel': df.ar.str.contains(TATWEEL, regex=False),
    'presentation_forms': df.ar.str.contains(PRESENTATION_FORM),
    'eastern_digits': df.ar.str.contains(EASTERN_DIGIT),
    'western_digits': df.ar.str.contains(r'[0-9]'),
    'html': df.ar.str.contains(HTML_TAG) | df.en.str.contains(HTML_TAG),
    'extra_whitespace': df.ar.str.contains(EXTRA_WS) | df.en.str.contains(EXTRA_WS),
    'latin_punct_in_ar': df.ar.str.contains(r'[,;?]') & df.ar.map(lambda s: arabic_share(s) > 0),
    'brackets': df.ar.str.contains(r'[()\[\]]'),
})
sc = script.groupby('source').sum()
S['script'] = {k: {c: int(v) for c, v in r.items()} for k, r in sc.iterrows()}
df['arabic_share'] = df.ar.map(arabic_share)
S['arabic_share_lt_1'] = df[df.arabic_share < 1].groupby('source').size().astype(int).to_dict()
sc

,diacritics,tatweel,presentation_forms,eastern_digits,western_digits,html,extra_whitespace,latin_punct_in_ar,brackets
source,,,,,,,,,
darby,0,0,0,0,0,0,0,0,0
gist,139,3,0,0,45,0,3,0,4
ms,888,329,1,1,414,0,0,6,233


## 7. Several translations packed into one cell

In [10]:
df['variants'] = df.ar.map(split_packed)
df['n_variants'] = df.variants.map(len)
packed = df[df.n_variants > 1]
S['packed'] = packed.groupby('source').size().astype(int).to_dict()
packed[['source', 'en', 'ar', 'n_variants']].head(20)

,source,en,ar,n_variants
65,gist,A/B test,اختبار A/B,2
4545,gist,time/space complexity,تعقيد الزمن/المكان,2
4579,gist,train / test / dev split,تقسيم التدريب / الاختبار / التحقق,3
4582,gist,train/test,التدريب/الاختبار,2
4904,ms,country/region-specific,خاص بالبلد/الإقليم,2
4907,ms,A/V Conferencing Edge service,خدمة Conferencing Edge للصوت/الفيديو,2
5084,ms,client/server computing,نظام حساب تطبيقات عميل/خادم,2
5085,ms,client/server architecture,هندسة العميل/الخادم,2
5121,ms,OLE/DDE link,ارتباط OLE/DDE,2
5729,ms,Transmission Control Protocol/Internet Protocol,برتوكول التحكم في الإرسال/بروتوكول الإنترنت,2


## 8. Conflicts: the same English term with different Arabic across sources

Compared with `ar_compare_key` (no diacritics, unified alef, ى and ة, no leading ال), so المصدر and مصدر agree.

In [11]:
from data.text import ar_compare_key
df['akey'] = df.ar.map(ar_compare_key)
multi = df.groupby('key').filter(lambda d: d.source.nunique() > 1)
conf = multi.groupby('key').filter(lambda d: d.akey.nunique() > 1)
S['conflicting_keys'] = int(conf.key.nunique())
S['agreeing_cross_source_keys'] = int(multi.key.nunique() - conf.key.nunique())
pairs = Counter()
for _, d in conf.groupby('key'):
    srcs = sorted(set(d.source))
    for i in range(len(srcs)):
        for j in range(i + 1, len(srcs)):
            pairs[f'{srcs[i]}-{srcs[j]}'] += 1
S['conflict_pairs'] = dict(pairs)
darby_conf = conf[conf.key.isin(conf[conf.source == 'darby'].key)].sort_values(['key', 'source'])
S['darby_conflicts'] = darby_conf[['key', 'source', 'ar']].to_dict('records')
print(pairs)
darby_conf[['key', 'source', 'en', 'ar']]

Counter({'gist-ms': 136, 'darby-ms': 13, 'darby-gist': 5})


,key,source,en,ar
0,AI,darby,AI,الذكاء الاصطناعي
67,AI,gist,AI,AI
17798,AI,ms,AI,الذكاء الاصطناعي
34,API,darby,API,واجهة برمجة التطبيقات
69,API,gist,API,API
28,binary,darby,Binary,النظام الثنائي
5398,binary,ms,binary,ثنائي
14181,binary,ms,binary,ثنائي
27,chaining,darby,Chaining,تسلسل الأوامر
5599,chaining,ms,chaining,تسلسل


## 9. Length distributions

In [12]:
df['en_chars'] = df.en.str.len(); df['en_words'] = df.en.str.split().map(len)
df['ar_chars'] = df.ar.str.len(); df['ar_words'] = df.ar.str.split().map(len)
lens = df.groupby('source')[['en_chars', 'en_words', 'ar_chars', 'ar_words']].describe().T
S['length'] = {src: {'en_words_max': int(d.en_words.max()), 'en_chars_p99': float(d.en_chars.quantile(.99)), 'single_word_share': round(float((d.en_words == 1).mean()), 3)} for src, d in df.groupby('source')}
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for src, d in df.groupby('source'):
    axes[0].hist(d.en_words.clip(upper=8), bins=range(1, 10), alpha=.55, label=src, density=True)
    axes[1].hist(d.en_chars.clip(upper=60), bins=30, alpha=.55, label=src, density=True)
axes[0].set_title('English words per term (8 = 8 or more)'); axes[1].set_title('English characters per term (60 = 60 or more)')
for a in axes: a.legend()
fig.tight_layout(); fig.savefig(REPORTS / 'eda_lengths.png', dpi=120)
junk = df[(df.en.str.strip().str.len() <= 1) | df.en.str.fullmatch(r'[\d\W_]+') | (df.en_words > 8)]
S['junk_candidates'] = junk.groupby('source').size().astype(int).to_dict()
display(junk[['source', 'en', 'ar']].head(20))

,source,en,ar
8863,ms,Develop Skills Needs - Hiring Plan - Hiring Fo...,تطوير الاحتياجات المهارية - خطة التوظيف - فرص ...
12069,ms,Microsoft Visual Studio Tools for the Microsof...,أدوات Microsoft Visual Studio لـ Microsoft Off...
12769,ms,Preparation of an Opening Balance Sheet at the...,تجهيز الميزانية العمومية الافتتاحية في تاريخ ا...
13201,ms,"Microsoft Office Compatibility Pack for Word, ...",حزمة التوافق من Microsoft Office لتنسيقات ملفا...
15028,ms,Principles for the data access and the testabi...,مبادئ الوصول إلى البيانات وقابلية المستندات ال...
15869,ms,1980 United Nations Convention on Contracts fo...,اتفاقية الأمم المتحدة المتعلقة بعقود البيع الد...
16687,ms,Allow Users to Log in to Spam Quarantine option,السماح للمستخدم بتسجيل الدخول إلى خيار عزل الب...
16881,ms,IEEE Standard Protocol for Authentication in H...,بروتوكول IEEE القياسي لمصادقة مرفقات المضيف لأ...
17687,ms,"Microsoft Office Compatibility Pack for Word, ...",حزمة التوافق من Microsoft Office لتنسيقات ملفا...
17846,ms,Business Contact Manager for Microsoft Outlook...,أداة قاعدة بيانات Business Contact Manager for...


## 10. Coverage and noise in real text

The term finder is run on the English text of our 7 lessons plus the demo email, once per source. Coverage: which terms appear. Noise: terms highlighted per 100 words. A source that marks every second word is useless for highlighting.

In [13]:
lesson_text = []
for p in sorted((ROOT / 'content').glob('day-*.json')):
    for b in json.loads(p.read_text(encoding='utf-8'))['blocks']:
        for v in (b.get('en'), b.get('text', {}).get('en') if isinstance(b.get('text'), dict) else None):
            if isinstance(v, str): lesson_text.append(re.sub(r'\[\[([^|\]]+)(?:\|[^\]]+)?\]\]', r'\1', v))
demo = 'Before you send the summary to the client, check it for hallucinations. Give the model more context in your prompt and ask it to cite the source paragraph.'
corpus = '\n'.join(lesson_text + [demo])
n_words = len(re.findall(r'[A-Za-z]+', corpus))
noise, hits_by = {}, {}
for src, d in df.groupby('source'):
    f = TermFinder([Term(id=f'{src}:{r}', en=e, ar=a, source=src) for r, e, a in zip(d.row_id, d.en, d.ar) if e.strip()])
    hits = f.find(corpus)
    hits_by[src] = Counter(h.text.lower() for h in hits)
    noise[src] = {'matches': len(hits), 'per_100_words': round(100 * len(hits) / n_words, 1), 'distinct_terms': len(hits_by[src])}
S['corpus_words'] = n_words; S['noise'] = noise
display(pd.DataFrame(noise).T)
for src in ('gist', 'ms'):
    print(src, 'most frequent matches:', hits_by[src].most_common(25))

,matches,per_100_words,distinct_terms
darby,91.0,7.5,33.0
gist,53.0,4.4,17.0
ms,383.0,31.5,183.0


gist most frequent matches: [('ai', 14), ('prompt', 9), ('policy', 5), ('source', 3), ('template', 3), ('confidence', 2), ('output', 2), ('constraint', 2), ('iteration', 2), ('dataset', 2), ('placeholders', 2), ('bias', 2), ('large language model', 1), ('state', 1), ('constraints', 1), ('training', 1), ('templates', 1)]
ms most frequent matches: [('ai', 14), ('use', 13), ('write', 11), ('prompt', 9), ('text', 8), ('check', 8), ('client', 7), ('summary', 6), ('facts', 6), ('manager', 5), ('words', 5), ('next', 5), ('point', 5), ('about', 5), ('answer', 5), ('notes', 5), ('data', 5), ('formula', 5), ('message', 4), ('meeting', 4), ('context', 4), ('column', 4), ('policy', 3), ('colleague', 3), ('writes', 3)]


In [14]:
# Which MS matches are everyday words? Single-word MS terms that our lesson text uses most.
ms_single = df[(df.source == 'ms') & (df.en_words == 1)]
S['ms_single_word_terms'] = int(len(ms_single))
for label, sub in (('ms multi-word only', df[(df.source == 'ms') & (df.en_words > 1)]), ('ms single-word only', ms_single)):
    f = TermFinder([Term(id=f'ms:{r}', en=e, ar=a, source='ms') for r, e, a in zip(sub.row_id, sub.en, sub.ar) if e.strip()])
    hits = f.find(corpus)
    noise[label] = {'matches': len(hits), 'per_100_words': round(100 * len(hits) / n_words, 1), 'distinct_terms': len({h.text.lower() for h in hits})}
    print(label, noise[label], Counter(h.text.lower() for h in hits).most_common(15))
S['noise'] = noise
demo_hits = {src: [h.text for h in TermFinder([Term(id=f'{src}:{r}', en=e, ar=a, source=src) for r, e, a in zip(d.row_id, d.en, d.ar) if e.strip()]).find(demo)] for src, d in df.groupby('source')}
S['demo_email_matches'] = demo_hits
demo_hits

ms multi-word only {'matches': 6, 'per_100_words': 0.5, 'distinct_terms': 5} [('data policy', 2), ('language model', 1), ('my notes', 1), ('check names', 1), ('id numbers', 1)]
ms single-word only {'matches': 387, 'per_100_words': 31.9, 'distinct_terms': 180} [('ai', 14), ('use', 13), ('write', 11), ('prompt', 9), ('check', 9), ('text', 8), ('data', 7), ('client', 7), ('summary', 6), ('facts', 6), ('notes', 6), ('manager', 5), ('policy', 5), ('words', 5), ('next', 5)]


{'darby': ['summary', 'hallucinations', 'context', 'prompt', 'source'],
 'gist': ['prompt', 'source'],
 'ms': ['send',
  'summary',
  'client',
  'check',
  'model',
  'more',
  'context',
  'prompt',
  'source',
  'paragraph']}

## 11. Summary and report

In [15]:
(REPORTS / 'eda_summary.json').write_text(json.dumps(S, ensure_ascii=False, indent=1, default=str), encoding='utf-8')
r = S
lines = [
    '# Term bank EDA: summary',
    '',
    'Generated by `notebooks/eda.ipynb` from the raw snapshot pinned in `data/manifest.json`. Numbers are recomputed on every run.',
    '',
    '## Rows',
    '',
    '| Source | Rows | Empty English | Empty Arabic | Acronym terms | Packed variants |',
    '|---|---|---|---|---|---|',
]
for s in ('darby', 'gist', 'ms'):
    lines.append(f"| {s} | {r['rows'][s]} | {r['empty'][s]['en_empty']} | {r['empty'][s]['ar_empty']} | {r['acronym_terms'].get(s, 0)} | {r['packed'].get(s, 0)} |")
lines += [
    '',
    f"All sources: {r['rows_total']} rows, {r['unique_keys_all']} unique matching keys.",
    '',
    '## Duplicates',
    '',
    '| Source | Exact (en, ar) pairs | Same English | Same English ignoring case | Same matching key |',
    '|---|---|---|---|---|',
] + [f"| {s} | {d['exact_pairs']} | {d['exact_en']} | {d['case_insensitive_en']} | {d['same_key']} |" for s, d in r['duplicates_within'].items()] + [
    '',
    f"Across sources (same key): Darby and GIST {r['overlap']['darby_gist']}, Darby and MS {r['overlap']['darby_ms']}, GIST and MS {r['overlap']['gist_ms']}, all three {r['overlap']['all_three']}. "
    f"Keys that differ only by case (kept apart, like IT and it): {r['case_split_keys']}.",
    '',
    '## Untranslated or partly translated Arabic',
    '',
    '| Source | Arabic same as English | Latin only | Acronym only | Mixed Arabic and Latin |',
    '|---|---|---|---|---|',
] + [f"| {s} | {d['ar_same_as_en']} | {d['ar_latin_only']} | {d['ar_acronym_only']} | {d['ar_mixed_script']} |" for s, d in r['untranslated'].items()] + [
    '',
    f"Acronym terms whose Arabic has no Arabic meaning (Latin or acronym only): {r['acronyms_without_arabic_meaning']}. Acronym terms with mixed Arabic and Latin: {r['acronyms_with_mixed_arabic']}.",
    f"Short-form patterns in English: {', '.join(f'{k} {v}' for k, v in sorted(r['acronym_kinds'].items()))}.",
    '',
    '## Script issues in Arabic',
    '',
    '| Source | ' + ' | '.join(next(iter(r['script'].values())).keys()) + ' |',
    '|---|' + '---|' * len(next(iter(r['script'].values()))),
] + [f"| {s} | " + ' | '.join(str(v) for v in d.values()) + ' |' for s, d in r['script'].items()] + [
    '',
    '## Conflicts across sources',
    '',
    f"{r['conflicting_keys']} English keys have different Arabic in different sources ({', '.join(f'{k}: {v}' for k, v in r['conflict_pairs'].items())}); {r['agreeing_cross_source_keys']} cross-source keys agree. "
    'Conflicts with our own glossary (ours wins in stage 2):',
    '',
] + [f"- `{c['key']}` {c['source']}: {c['ar']}" for c in r['darby_conflicts']] + [
    '',
    '## Length and junk',
    '',
] + [f"- {s}: single-word share {d['single_word_share']}, longest English term {d['en_words_max']} words, 99th percentile {d['en_chars_p99']:.0f} characters." for s, d in r['length'].items()] + [
    f"- Junk candidates (one character, no letters, or more than 8 words): {r['junk_candidates']}.",
    '',
    '![Lengths](eda_lengths.png)',
    '',
    '## Coverage and noise in real text',
    '',
    f"Term finder run on the English text of the 7 lessons and the demo email ({r['corpus_words']} words), one source at a time:",
    '',
    '| Source | Matches | Per 100 words | Distinct terms |',
    '|---|---|---|---|',
] + [f"| {s} | {d['matches']} | {d['per_100_words']} | {d['distinct_terms']} |" for s, d in r['noise'].items()] + [
    '',
    'Demo email matches: ' + '; '.join(f"{s}: {', '.join(v) or 'none'}" for s, v in r['demo_email_matches'].items()) + '.',
]
(REPORTS / 'eda.md').write_text('\n'.join(lines) + '\n', encoding='utf-8')
print('\n'.join(lines[:60]))

# Term bank EDA: summary

Generated by `notebooks/eda.ipynb` from the raw snapshot pinned in `data/manifest.json`. Numbers are recomputed on every run.

## Rows

| Source | Rows | Empty English | Empty Arabic | Acronym terms | Packed variants |
|---|---|---|---|---|---|
| darby | 40 | 0 | 0 | 2 | 0 |
| gist | 4844 | 0 | 0 | 47 | 4 |
| ms | 22061 | 0 | 0 | 232 | 104 |

All sources: 26945 rows, 24052 unique matching keys.

## Duplicates

| Source | Exact (en, ar) pairs | Same English | Same English ignoring case | Same matching key |
|---|---|---|---|---|
| darby | 0 | 0 | 0 | 0 |
| gist | 0 | 0 | 0 | 16 |
| ms | 1082 | 1964 | 2422 | 2642 |

Across sources (same key): Darby and GIST 12, Darby and MS 27, GIST and MS 203, all three 7. Keys that differ only by case (kept apart, like IT and it): 20.

## Untranslated or partly translated Arabic

| Source | Arabic same as English | Latin only | Acronym only | Mixed Arabic and Latin |
|---|---|---|---|---|
| darby | 0 | 0 | 0 | 0 |
| gist | 60 